# 01 - Cleaning & Validation

Combines all five leagues and all six stats into one canonical **team-perspective**
table (two rows per match), runs the integrity checks, and confirms the
distributional assumption behind each target family's objective.

In [1]:
# The package is installed editable (`pip install -e .`), so this works from any
# working directory -- no `os.getcwd()` gymnastics.
import fpp
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("fpp", fpp.__version__, "| targets:", fpp.TARGETS)

fpp 0.1.0 | targets: ('goals', 'shots', 'sot', 'corners')


## 1. Build the canonical table

Understat (goals, xG, npxG) joined to ESPN (shots, SOT, corners) on
`(league, home, away)` with a +/-1 day tolerance -- ESPN timestamps are UTC kickoff
and can fall either side of the local match date.

In [2]:
df = fpp.clean.build_clean_table(refresh_stats=False)
print(f"{len(df):,} team-rows  |  {df['fixture_id'].nunique():,} fixtures")
print(f"{df['date'].min().date()} -> {df['date'].max().date()}")
df.head(3)

  dropped 3 unplayed/incomplete rows
  ESPN stats joined to 20,807/21,634 matches (96.2%)
  11 league-season(s) below 95% ESPN coverage:
league_key    season  matches  with_stats  pct
      Bund 2021/2022      306         273 89.2
      Liga 2022/2023      380         357 93.9
      Liga      2627       16           5 31.2
     Ligue      2627        9           0  0.0
      Prem 2016/2017      380          18  4.7
      Prem 2021/2022      380         353 92.9
      Prem 2022/2023      380         297 78.2
      Prem      2627       10           0  0.0
     Serie 2022/2023      380         342 90.0
     Serie 2023/2024      380         344 90.5
     Serie      2627       10           0  0.0
  Saved 43,268 rows -> /Users/patrickknott/.cache/football_prediction/clean/team_matches_0bad3cddb17f.parquet
43,268 team-rows  |  21,634 fixtures
2014-08-08 -> 2026-08-24


,fixture_id,date,season,league,league_key,team,opponent,is_home,has_espn_stats,goals_for,goals_against,xg_for,xg_against,npxg_for,npxg_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,points_for,points_against,game_week,game_week_normalised,team_rest_days,team_matches_14d,opp_rest_days,opp_matches_14d,team_is_new_to_league,team_rank_delta,opp_is_new_to_league,opp_rank_delta
0,0,2014-08-08,2014/2015,Ligue 1,Ligue,Reims,Paris Saint Germain,1,1,2,2,1.367870,2.65538,1.367870,1.89529,9.0,16.0,3.0,6.0,1.0,5.0,1,1,1,0.008850,NaN,0.0,NaN,0.0,0,0,0,0
1,0,2014-08-08,2014/2015,Ligue 1,Ligue,Paris Saint Germain,Reims,0,1,2,2,2.655380,1.36787,1.895290,1.36787,16.0,9.0,6.0,3.0,5.0,1.0,1,1,1,0.008850,NaN,0.0,NaN,0.0,0,0,0,0
2,1,2014-08-09,2014/2015,Ligue 1,Ligue,Evian Thonon Gaillard,Caen,1,1,0,3,0.813737,1.23869,0.813737,1.23869,10.0,12.0,2.0,7.0,5.0,6.0,0,3,2,0.017699,NaN,0.0,NaN,0.0,0,0,0,0


## 2. Coverage

In [3]:
cov = fpp.clean.coverage_report(df.drop_duplicates("fixture_id"))
per_league = cov.groupby("league_key")[["matches", "with_stats"]].sum()
per_league["pct"] = (100 * per_league["with_stats"] / per_league["matches"]).round(2)
display(per_league)

,matches,with_stats,pct
league_key,,,
Bund,3672,3622,98.64
Liga,4576,4513,98.62
Ligue,4246,4173,98.28
Prem,4570,4042,88.45
Serie,4570,4457,97.53


## 3. Integrity checks

Three things must hold before anything downstream runs:

1. every fixture is exactly two rows
2. each team's *for* equals its opponent's *against*, for every stat
3. **no team name appears in two leagues in the same season**

Check 3 matters because the pooled walk-forward buffer is keyed by team name across
all five leagues -- a collision would silently merge two clubs' histories. It is
clean today, and becomes genuinely load-bearing once English lower tiers are added.

In [4]:
fpp.clean.check_integrity(df)
print("integrity checks passed")

teams = pd.concat([df["team"], df["opponent"]]).nunique()
print(f"{teams} distinct clubs, zero cross-league name collisions")

integrity checks passed
170 distinct clubs, zero cross-league name collisions


## 4. Dispersion - does the objective choice hold up?

`count:poisson` is right when variance ~ mean. Materially above that means
overdispersion, which needs `reg:tweedie` plus a Negative Binomial pmf.

Two views. The **marginal** ratio is the headline, but it overstates the case: some
of that spread is just the difference between good and bad teams. The
**conditional** ratio strips that out and is the honest test.

In [5]:
marg = fpp.clean.dispersion_report(df)
display(marg.pivot(index="league", columns="stat", values="var_over_mean"))

stat,corners,goals,shots,sot
league,,,,
Bund,1.598,1.157,2.109,1.446
Liga,1.585,1.125,2.030,1.384
Ligue,1.555,1.130,1.926,1.361
Prem,1.727,1.130,2.392,1.427
Serie,1.678,1.064,2.206,1.392


In [6]:
cond = fpp.clean.conditional_dispersion(df)
display(cond)

print("\nRatios near 1.0 => Poisson is adequate; well above 1.0 => Tweedie/NB earned.")
for r in cond.itertuples():
    verdict = "Poisson adequate" if r.residual_var_over_mean < 1.25 else "overdispersed -> Tweedie/NB"
    print(f"  {r.stat:8} conditional var/mean = {r.residual_var_over_mean:5.2f}   {verdict}")

,stat,n,groups,mean,residual_var,residual_var_over_mean
0,goals,43268,1248,1.384,1.382,0.998
1,shots,41614,1178,12.513,22.941,1.833
2,sot,41614,1178,4.338,5.250,1.210
3,corners,41614,1178,4.926,7.488,1.520



Ratios near 1.0 => Poisson is adequate; well above 1.0 => Tweedie/NB earned.
  goals    conditional var/mean =  1.00   Poisson adequate
  shots    conditional var/mean =  1.83   overdispersed -> Tweedie/NB
  sot      conditional var/mean =  1.21   Poisson adequate
  corners  conditional var/mean =  1.52   overdispersed -> Tweedie/NB


## 5. Missingness

Shots/SOT/corners can be absent where the ESPN join failed. They are flagged rather
than dropped: those fixtures are still perfectly usable for the goals model.

In [7]:
miss = df[[c for c in df.columns if c.endswith(("_for", "_against"))]].isna().sum()
display(miss[miss > 0].to_frame("missing"))
print(f"rows with full ESPN stats: {df['has_espn_stats'].mean() * 100:.2f}%")

,missing
shots_for,1654
shots_against,1654
sot_for,1654
sot_against,1654
corners_for,1654
corners_against,1654


rows with full ESPN stats: 96.18%
